# Siamese Network — Weather Anomaly Detection (Stage 6)
**Dataset:** Jena Climate · **Task:** Metric learning for weather anomaly detection  
**Builds on:** Stage 4 CNN architecture · Stage 5 autoencoder reconstruction error

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from utils.data_prep import load_and_split, scale_features, get_window_timestamps
from utils.sequencer import make_sequences
from utils.pairs import compute_reconstruction_mse, derive_anomaly_labels, build_pairs
from utils.models import build_shared_cnn, build_siamese, distance_metric
from utils.losses import contrastive_loss
from utils.evaluator import (
    plot_training_curves,
    calibrate_threshold,
    knn_anomaly_scores,
    evaluate_roc,
    plot_nn_comparison,
    plot_mse_distribution,
)

DATA_PATH  = Path('../output/processed_data.csv')
SCALER_PATH = Path('../output/scaler.pkl')
AE_PATH    = Path('../output/autoencoder.h5')
SIAMESE_PATH   = Path('../output/siamese.h5')
SHARED_CNN_PATH = Path('../output/shared_cnn.h5')

WINDOW_SIZE   = 24
EMBEDDING_DIM = 128
MARGIN        = 1.0

print('TF version:', tf.__version__)
plt.rcParams['figure.dpi'] = 100
tf.random.set_seed(42)
np.random.seed(42)

---
## Section 1 — Data Preparation & Sequence Construction

In [ ]:
train_df, val_df, test_df = load_and_split(DATA_PATH)

In [ ]:
X_tr, X_va, X_te, feat_scaler, feature_cols = scale_features(
    train_df, val_df, test_df, scaler_path=SCALER_PATH
)
N_FEATURES = X_tr.shape[1]

DEMO_FEAT_IDX  = feature_cols.index('T (degC)') if 'T (degC)' in feature_cols else 0
DEMO_FEAT_NAME = feature_cols[DEMO_FEAT_IDX]
print(f'Demo feature: "{DEMO_FEAT_NAME}" at index {DEMO_FEAT_IDX}')

In [ ]:
X_train_seq = make_sequences(X_tr, WINDOW_SIZE)
X_val_seq   = make_sequences(X_va, WINDOW_SIZE)
X_test_seq  = make_sequences(X_te, WINDOW_SIZE)

print(f'X_train_seq : {X_train_seq.shape}')
print(f'X_val_seq   : {X_val_seq.shape}')
print(f'X_test_seq  : {X_test_seq.shape}')

---
## Section 2 — Derive Anomaly Labels from Stage 5 Autoencoder

In [ ]:
autoencoder = tf.keras.models.load_model(str(AE_PATH), compile=False)
print('Autoencoder loaded from Stage 5')
autoencoder.summary()

In [ ]:
labels_train, labels_val, labels_test, ae_threshold = derive_anomaly_labels(
    autoencoder, X_train_seq, X_val_seq, X_test_seq, percentile=95
)

# MSE for distribution plot
mse_train_full = compute_reconstruction_mse(autoencoder, X_train_seq)
plot_mse_distribution(mse_train_full, ae_threshold)

---
## Section 3 — Construct Contrastive Pairs

In [ ]:
pairs_A_tr, pairs_B_tr, pair_labels_tr = build_pairs(
    X_train_seq, labels_train, neg_per_anomaly=3, seed=42
)
pairs_A_va, pairs_B_va, pair_labels_va = build_pairs(
    X_val_seq, labels_val, neg_per_anomaly=3, seed=0
)
pairs_A_te, pairs_B_te, pair_labels_te = build_pairs(
    X_test_seq, labels_test, neg_per_anomaly=3, seed=1
)

for split, pl in [('Train', pair_labels_tr), ('Val', pair_labels_va), ('Test', pair_labels_te)]:
    n_pos = (pl == 0).sum()
    n_neg = (pl == 1).sum()
    print(f'{split}: {len(pl):,} pairs  (positive={n_pos:,}, negative={n_neg:,})')

---
## Section 4 — Shared CNN Subnet (Stage 4 architecture + embedding)

In [ ]:
shared_cnn = build_shared_cnn(WINDOW_SIZE, N_FEATURES, embedding_dim=EMBEDDING_DIM)
shared_cnn.summary()

---
## Section 5 — Siamese Network & Contrastive Loss Training

In [ ]:
siamese = build_siamese(shared_cnn, WINDOW_SIZE, N_FEATURES)
siamese.summary()

siamese.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=contrastive_loss(margin=MARGIN),
    metrics=[distance_metric(threshold=0.5)],
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(SIAMESE_PATH), monitor='val_loss', save_best_only=True, verbose=0
    ),
]

history = siamese.fit(
    [pairs_A_tr, pairs_B_tr],
    pair_labels_tr,
    validation_data=([pairs_A_va, pairs_B_va], pair_labels_va),
    epochs=200,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plot_training_curves(history)

---
## Section 6 — Distance Threshold Calibration (Validation Set)

In [ ]:
tau = calibrate_threshold(
    siamese,
    pairs_A_va, pairs_B_va, pair_labels_va,
    n_thresholds=200,
)
print(f'Chosen threshold τ = {tau:.4f}')

---
## Section 7 — Anomaly Detection on Test Set (k-NN scoring)

In [ ]:
# Use only normal training windows as the reference set
X_train_normal = X_train_seq[labels_train == 0]
print(f'Normal training windows: {X_train_normal.shape[0]:,}')

anomaly_scores = knn_anomaly_scores(
    shared_cnn, X_test_seq, X_train_normal, k=10
)
print(f'Anomaly scores — min={anomaly_scores.min():.4f}  max={anomaly_scores.max():.4f}  '
      f'mean={anomaly_scores.mean():.4f}')

In [ ]:
# Use the pair-distance τ as the knn-score threshold (both are distances in embedding space)
auc = evaluate_roc(anomaly_scores, labels_test, tau=tau)

In [ ]:
plot_nn_comparison(
    shared_cnn,
    X_test_seq, X_train_normal,
    test_labels=labels_test,
    feature_idx=DEMO_FEAT_IDX,
    feature_name=DEMO_FEAT_NAME,
    n_examples=3,
)

---
## Section 8 — Full Cross-Stage Comparison Table

In [ ]:
# Fill in metrics from previous stages here
# These values should be updated with actual results after each stage runs
comparison = pd.DataFrame([
    {'Stage': 2, 'Model': 'MLP',           'Task': 'Regression',              'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 2, 'Model': 'DNN',           'Task': 'Regression',              'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 3, 'Model': 'LSTM 1-Layer',  'Task': 'Regression',              'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 3, 'Model': 'LSTM 2-Layer',  'Task': 'Regression',              'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 4, 'Model': 'Persistence',   'Task': 'Regression (baseline)',   'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 4, 'Model': 'CNN Simple',    'Task': 'Regression',              'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 4, 'Model': 'CNN Multi-Scale','Task': 'Regression',             'Metric': 'Test RMSE (°C)', 'Score': '—'},
    {'Stage': 5, 'Model': 'Autoencoder',   'Task': 'Anomaly Detection',       'Metric': 'Anomaly Rate (%)', 'Score': f'{labels_train.mean()*100:.1f}'},
    {'Stage': 6, 'Model': 'Siamese Network','Task': 'Anomaly Detection',      'Metric': 'AUC / F1', 'Score': f'{auc:.3f}'},
])
print(comparison.to_string(index=False))

**Best for forecasting:** The CNN and LSTM models capture temporal structure effectively. LSTMs are better for long-range dependencies while CNNs are faster and more parameter-efficient. Multi-scale CNN combines short and long receptive fields to handle different weather cycles.

**Best for anomaly detection:** The autoencoder detects anomalies by reconstruction difficulty — patterns the model has not learned to compress well. The Siamese network uses metric learning to push anomalous windows away from normal ones in embedding space, enabling cleaner separation.

**Architectural takeaways:** Recurrence (LSTM) is suited for sequential dependencies with long-range context. Convolution (CNN) excels at local pattern extraction with fewer parameters. Metric learning (Siamese) leverages label structure to shape the embedding space explicitly, outperforming unsupervised reconstruction when labeled anomaly examples are available.

---
## Section 9 — Save Artifacts

In [ ]:
from utils.models import L2Normalize, EuclideanDistance

shared_cnn.save(str(SHARED_CNN_PATH))

_custom = {'L2Normalize': L2Normalize, 'EuclideanDistance': EuclideanDistance}
siamese_reloaded    = tf.keras.models.load_model(str(SIAMESE_PATH),    compile=False, custom_objects=_custom)
shared_cnn_reloaded = tf.keras.models.load_model(str(SHARED_CNN_PATH), compile=False, custom_objects=_custom)

# Verify shapes
dist_sample = siamese_reloaded.predict([X_test_seq[:4], X_test_seq[4:8]], verbose=0)
emb_sample  = shared_cnn_reloaded.predict(X_test_seq[:4], verbose=0)
print(f'Siamese output shape   : {dist_sample.shape}  (expected (4,1))')
print(f'SharedCNN output shape : {emb_sample.shape}  (expected (4, {EMBEDDING_DIM}))')

for p in [SIAMESE_PATH, SHARED_CNN_PATH]:
    size = p.stat().st_size if p.exists() else -1
    print(f'{p.name}: {size:,} bytes')

# Save metrics summary
reports_dir = Path('../reports')
reports_dir.mkdir(exist_ok=True)
comparison.to_csv(reports_dir / 'metrics_summary.csv', index=False)
print('metrics_summary.csv saved')